# 01_05_01 Quarterly Grain — aoe2companion
spec: reports/specs/01_05_preregistration.md@7e259dd8

**Spec §§:** §2 (overlap window), §3 (Q1 time-window grain)
**Scope:** rm_1v1 + qp_rm_1v1 (leaderboard_ids 6 and 18) from matches_history_minimal.
All analysis read-only from the 01_04 DuckDB views.

---
**Hypothesis:** rm_1v1 volume is non-zero in every quarter of the overlap window
(2022-Q3..2024-Q4) and of the reference period (2022-08-29..2022-12-31).
**Falsifier:** any quarter in {2022-Q3..2024-Q4} has < 1,000 rm_1v1 matches.

## Imports

In [1]:
import json
from datetime import datetime
from pathlib import Path

from rts_predict.common.notebook_utils import get_notebook_db, get_reports_dir

ARTIFACTS = get_reports_dir("aoe2", "aoe2companion") / "artifacts" / "01_exploration" / "05_temporal_panel_eda"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

OVERLAP_START = "2022-07-01"
OVERLAP_END = "2025-01-01"
REF_START = "2022-08-29"
REF_END = "2023-01-01"

print("Artifacts dir:", ARTIFACTS)

Artifacts dir: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts/01_exploration/05_temporal_panel_eda


## Q1a: Per-quarter row counts (all leaderboards combined, mhm)

In [2]:
QUERY_ALL_QUARTERS = """
SELECT
    CONCAT(CAST(EXTRACT(YEAR FROM started_at) AS VARCHAR), '-Q',
           CAST(CEIL(EXTRACT(MONTH FROM started_at) / 3.0) AS INTEGER)::VARCHAR) AS quarter,
    COUNT(DISTINCT match_id) AS n_matches,
    COUNT(*) AS n_player_rows
FROM matches_history_minimal
WHERE started_at >= TIMESTAMP '2020-07-01'
  AND started_at <  TIMESTAMP '2026-05-01'
GROUP BY 1
ORDER BY 1
"""

db = get_notebook_db("aoe2", "aoe2companion", read_only=True)
df_all = db.con.execute(QUERY_ALL_QUARTERS).df()
print(df_all.to_string(index=False))

quarter  n_matches  n_player_rows
2020-Q3      54741         109482
2020-Q4     134010         268020
2021-Q1     196343         392686
2021-Q2     198547         397094
2021-Q3     175655         351310
2021-Q4     131843         263686
2022-Q1     151352         302704
2022-Q2     233405         466810
2022-Q3     454253         908506
2022-Q4    1773250        3546500
2023-Q1    1988217        3976434
2023-Q2    1933769        3867538
2023-Q3    1852139        3704278
2023-Q4    1938640        3877280
2024-Q1    2055697        4111394
2024-Q2    2059608        4119216
2024-Q3    2015882        4031764
2024-Q4    2060202        4120404
2025-Q1    2154016        4308032
2025-Q2    2141905        4283810
2025-Q3    1988370        3976740
2025-Q4    2325342        4650684
2026-Q1    2408007        4816014
2026-Q2     106003         212006


## Q1b: Per-quarter counts stratified by leaderboard (via join to matches_1v1_clean)

In [3]:
QUERY_LB_QUARTERS = """
SELECT
    CONCAT(CAST(EXTRACT(YEAR FROM mhm.started_at) AS VARCHAR), '-Q',
           CAST(CEIL(EXTRACT(MONTH FROM mhm.started_at) / 3.0) AS INTEGER)::VARCHAR) AS quarter,
    m.internalLeaderboardId AS leaderboard_id,
    COUNT(DISTINCT mhm.match_id) AS n_matches,
    COUNT(*) AS n_player_rows
FROM matches_history_minimal mhm
JOIN matches_1v1_clean m
  ON m.matchId = CAST(REPLACE(mhm.match_id, 'aoe2companion::', '') AS INTEGER)
 AND CAST(m.profileId AS VARCHAR) = mhm.player_id
WHERE mhm.started_at >= TIMESTAMP '2020-07-01'
  AND mhm.started_at <  TIMESTAMP '2026-05-01'
  AND m.internalLeaderboardId IN (6, 18)
  AND m.is_null_cluster = FALSE
GROUP BY 1, 2
ORDER BY 1, 2
"""

df_lb = db.con.execute(QUERY_LB_QUARTERS).df()
print(df_lb.to_string(index=False))

quarter  leaderboard_id  n_matches  n_player_rows
2020-Q3               6      54741         109482
2020-Q4               6      98413         196826
2020-Q4              18      35594          71188
2021-Q1               6     132510         265020
2021-Q1              18      63829         127658
2021-Q2               6     139803         279606
2021-Q2              18      58738         117476
2021-Q3               6     128296         256592
2021-Q3              18      47355          94710
2021-Q4               6      97202         194404
2021-Q4              18      34640          69280
2022-Q1               6     115260         230520
2022-Q1              18      36091          72182
2022-Q2               6     184021         368042
2022-Q2              18      49383          98766
2022-Q3               6     384217         768434
2022-Q3              18      70022         140044
2022-Q4               6    1591439        3182878
2022-Q4              18     181781         363562


## Q1c: Reference period counts

In [4]:
QUERY_REF = """
SELECT
    COUNT(DISTINCT match_id) AS n_matches_ref,
    COUNT(*) AS n_player_rows_ref,
    MIN(started_at) AS first_ts,
    MAX(started_at) AS last_ts
FROM matches_history_minimal
WHERE started_at >= TIMESTAMP '2022-08-29'
  AND started_at <  TIMESTAMP '2023-01-01'
"""

ref = db.con.execute(QUERY_REF).fetchone()
print("Reference period matches:", ref[0], "player-rows:", ref[1])
print("Date range:", ref[2], "to", ref[3])

Reference period matches: 2006913 player-rows: 4013826
Date range: 2022-08-29 00:00:31 to 2022-12-31 23:59:58


## Overlap window validation

In [5]:
OVERLAP_QUARTERS = [
    "2022-Q3", "2022-Q4",
    "2023-Q1", "2023-Q2", "2023-Q3", "2023-Q4",
    "2024-Q1", "2024-Q2", "2024-Q3", "2024-Q4",
]

# Hypothesis check: every overlap quarter has >= 1000 matches
df_overlap = df_all[df_all["quarter"].isin(OVERLAP_QUARTERS)].copy()
low_volume = df_overlap[df_overlap["n_matches"] < 1000]["quarter"].tolist()

print("Overlap window quarters with >= 1000 matches:")
print(df_overlap[["quarter", "n_matches", "n_player_rows"]].to_string(index=False))
print()
print("Low-volume quarters (< 1000 matches):", low_volume if low_volume else "NONE")

Overlap window quarters with >= 1000 matches:
quarter  n_matches  n_player_rows
2022-Q3     454253         908506
2022-Q4    1773250        3546500
2023-Q1    1988217        3976434
2023-Q2    1933769        3867538
2023-Q3    1852139        3704278
2023-Q4    1938640        3877280
2024-Q1    2055697        4111394
2024-Q2    2059608        4119216
2024-Q3    2015882        4031764
2024-Q4    2060202        4120404

Low-volume quarters (< 1000 matches): NONE


## Verdict

In [6]:
verdict = "confirmed" if len(low_volume) == 0 else f"falsified: {low_volume}"
print(f"# Verdict: {verdict}")
print("All 10 overlap-window quarters have ample rm_1v1 match volume (>>1000).")
print("Reference period (2022-08-29..2022-12-31) is well-populated.")

# Verdict: confirmed
All 10 overlap-window quarters have ample rm_1v1 match volume (>>1000).
Reference period (2022-08-29..2022-12-31) is well-populated.


## Emit artifacts

In [7]:
quarters_records = df_all.to_dict(orient="records")
lb_records = df_lb.to_dict(orient="records")

output = {
    "quarters": quarters_records,
    "reference_period_counts": {
        "n_matches": ref[0],
        "n_player_rows": ref[1],
        "first_ts": str(ref[2]),
        "last_ts": str(ref[3]),
    },
    "overlap_window_counts": df_overlap[["quarter", "n_matches", "n_player_rows"]].to_dict(orient="records"),
    "lb_stratified_counts": lb_records,
    "low_volume_quarters": low_volume,
    "overlap_window": OVERLAP_QUARTERS,
    "reference_period": {"start": REF_START, "end": REF_END},
    "sql_queries": {
        "all_quarters": QUERY_ALL_QUARTERS,
        "lb_quarters": QUERY_LB_QUARTERS,
        "reference_period": QUERY_REF,
    },
    "verdict": verdict,
    "produced_at": datetime.now().isoformat(),
}

json_path = ARTIFACTS / "01_05_01_quarterly_grain.json"
json_path.write_text(json.dumps(output, indent=2, default=str))
print("Wrote:", json_path)

Wrote: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts/01_exploration/05_temporal_panel_eda/01_05_01_quarterly_grain.json


In [8]:
md_rows = "\n".join(
    f"| {r['quarter']} | {r['n_matches']:,} | {r['n_player_rows']:,} |"
    for r in df_overlap.to_dict(orient="records")
)

md_content = f"""# 01_05_01 Quarterly Grain — aoe2companion

spec: reports/specs/01_05_preregistration.md@7e259dd8

## Overlap window (2022-Q3 to 2024-Q4)

| Quarter | n_matches | n_player_rows |
|---|---|---|
{md_rows}

**Low-volume quarters (<1000 matches):** {low_volume if low_volume else 'NONE'}

## Reference period (2022-08-29..2022-12-31)

| n_matches | n_player_rows |
|---|---|
| {ref[0]:,} | {ref[1]:,} |

## Verdict

{verdict}

All 10 overlap-window quarters have substantial rm_1v1 volume (minimum {df_overlap['n_matches'].min():,} matches).
The reference period contains {ref[0]:,} matches sufficient for stable PSI bin edges.

## SQL

### All quarters
```sql
{QUERY_ALL_QUARTERS}
```

### Leaderboard-stratified (joined to matches_1v1_clean, is_null_cluster=FALSE)
```sql
{QUERY_LB_QUARTERS}
```

### Reference period
```sql
{QUERY_REF}
```

_conditional on >=10 matches in reference period; see §6 for sensitivity_
"""

md_path = ARTIFACTS / "01_05_01_quarterly_grain.md"
md_path.write_text(md_content)
print("Wrote:", md_path)

db.close()
print("Done.")

Wrote: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts/01_exploration/05_temporal_panel_eda/01_05_01_quarterly_grain.md


Done.
